<a href="https://colab.research.google.com/github/hoshi-3104-com/california-house-price-competition/blob/main/notebooks/05_house_price_parameter_tuning_lgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Kaggle APIを利用できるようにするため、kaggle.jsonからusernameとkeyを設定する
import os
import json
f = open("/content/drive/MyDrive/house_price_competion_hoshino/kaggle.json", 'r') # ディレクトリは必要に応じて変更してください
json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# 前処理(正規化・標準化)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.linear_model import LinearRegression

# 精度評価
from sklearn.metrics import mean_squared_error

from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/train_data_raw.csv')
test = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/test_data_raw.csv')
sample = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/sample_raw.csv')

# 2. データの切り分け
X_all_train = train.drop(["Price", "id"], axis=1)
y_all_train = train["Price"]
X_test = test.drop(["id"], axis=1) if "id" in test.columns else test

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.7 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*X does not have valid feature names.*')
import optuna
from lightgbm import LGBMRegressor

# 1. 各Foldで事前にRFE（特徴量選択）を実行し、データを保存（高速化のため）
folds_num = 5
kf = KFold(n_splits=folds_num, shuffle=True, random_state=42)
cv_datasets = []

print("RFEによる特徴量選択を開始します...")
for cv_train_idx, cv_val_idx in kf.split(X_all_train, y_all_train):
    X_cv_train, y_cv_train = X_all_train.iloc[cv_train_idx], y_all_train.iloc[cv_train_idx]
    X_cv_val, y_cv_val = X_all_train.iloc[cv_val_idx], y_all_train.iloc[cv_val_idx]

    rfe = RFE(estimator=LGBMRegressor(random_state=42, verbose=-1), step=1)
    rfe.fit(X_cv_train, y_cv_train)

    cv_datasets.append({
        "X_train": rfe.transform(X_cv_train), "y_train": y_cv_train,
        "X_val": rfe.transform(X_cv_val), "y_val": y_cv_val,
        "X_test": rfe.transform(X_test), "val_idx": cv_val_idx
    })

# Optunaの目的関数（feature_fraction → max_depth に変更）
def objective(trial):
    params = {
        "random_state": 42,
        "verbose": -1,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12), # 追加：木の最大深さ
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    scores = []
    for data in cv_datasets:
        model = LGBMRegressor(**params)
        model.fit(data["X_train"], data["y_train"])
        preds = np.clip(model.predict(data["X_val"]), 0, 5.00001)
        scores.append(np.sqrt(mean_squared_error(data["y_val"], preds)))
    return np.mean(scores)

# パラメータ探索の実行
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
print(f"\n★ ベストパラメータ: {best_params}")

# 4. 最適パラメータを用いて最終的な予測（OOFとテスト）を作成
oof_predictions = np.zeros(len(X_all_train))
test_predictions = np.zeros(len(X_test))
final_params = {"random_state": 42, "verbose": -1, **best_params}

for data in cv_datasets:
    model = LGBMRegressor(**final_params)
    model.fit(data["X_train"], data["y_train"])

    oof_predictions[data["val_idx"]] = np.clip(model.predict(data["X_val"]), 0, 5.00001)
    test_predictions += np.clip(model.predict(data["X_test"]), 0, 5.00001) / folds_num

# 5. スコア評価と提出ファイル作成
overall_oof_score = np.sqrt(mean_squared_error(y_all_train, oof_predictions))
print(f"全体のOOFスコア (RMSE): {overall_oof_score:.4f}")

sample["Price"] = test_predictions
# sample.to_csv('/content/drive/MyDrive/house_price_competion_hoshino/submission_optuna.csv', index=False)

RFEによる特徴量選択を開始します...


[I 2026-05-21 03:11:50,716] A new study created in memory with name: no-name-8e1e2175-de0b-4b6f-905f-190a531fb411
[I 2026-05-21 03:11:51,492] Trial 0 finished with value: 0.4917431118128833 and parameters: {'lambda_l1': 0.0026039710847556535, 'lambda_l2': 2.990107793830621e-08, 'num_leaves': 35, 'max_depth': 5, 'bagging_fraction': 0.7056816133292112, 'bagging_freq': 1, 'min_child_samples': 8}. Best is trial 0 with value: 0.4917431118128833.
[I 2026-05-21 03:11:52,484] Trial 1 finished with value: 0.49240305251606936 and parameters: {'lambda_l1': 0.00011602658199899, 'lambda_l2': 2.851300248570785e-06, 'num_leaves': 154, 'max_depth': 6, 'bagging_fraction': 0.5589642073588584, 'bagging_freq': 2, 'min_child_samples': 61}. Best is trial 0 with value: 0.4917431118128833.
[I 2026-05-21 03:11:53,207] Trial 2 finished with value: 0.4948402300559346 and parameters: {'lambda_l1': 0.0008666061132991414, 'lambda_l2': 4.121334190430836e-07, 'num_leaves': 130, 'max_depth': 5, 'bagging_fraction': 0.8


★ ベストパラメータ: {'lambda_l1': 0.0015528228924158874, 'lambda_l2': 1.2334242036151544e-07, 'num_leaves': 78, 'max_depth': 11, 'bagging_fraction': 0.7735839118397572, 'bagging_freq': 3, 'min_child_samples': 14}
全体のOOFスコア (RMSE): 0.4652


In [ ]:
# csvファイルの作成
sample.to_csv('submit_raw_cv_lgbm_optuna.csv',index=None)

In [ ]:
# 作成したファイルをKaggleに直接投稿
# -c:提出コンペ名
# -f：ファイル名
# -m：投稿コメント

!kaggle competitions submit -c ambl-california-housing -f submit_raw_cv_lgbm_optuna.csv -m "Yeah! I submit my file through the Google Colab!"

100% 94.0k/94.0k [00:00<00:00, 187kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing

#### 特徴量追加後、外れ値除去後のデータでハイパーパラメータチューニングしてみる

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/2_preprocess_train_data_add_feature.csv')
test = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/1_preprocess_test_data_add_feature.csv')
sample = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/sample_raw.csv')

# 2. データの切り分け
X_all_train = train.drop(["Price", "id"], axis=1)
y_all_train = train["Price"]
X_test = test.drop(["id"], axis=1) if "id" in test.columns else test

import warnings
warnings.filterwarnings('ignore', message='.*X does not have valid feature names.*')
import optuna
from lightgbm import LGBMRegressor

# 1. 各Foldで事前にRFE（特徴量選択）を実行し、データを保存（高速化のため）
folds_num = 5
kf = KFold(n_splits=folds_num, shuffle=True, random_state=42)
cv_datasets = []

print("RFEによる特徴量選択を開始します...")
for cv_train_idx, cv_val_idx in kf.split(X_all_train, y_all_train):
    X_cv_train, y_cv_train = X_all_train.iloc[cv_train_idx], y_all_train.iloc[cv_train_idx]
    X_cv_val, y_cv_val = X_all_train.iloc[cv_val_idx], y_all_train.iloc[cv_val_idx]

    rfe = RFE(estimator=LGBMRegressor(random_state=42, verbose=-1), step=1)
    rfe.fit(X_cv_train, y_cv_train)

    cv_datasets.append({
        "X_train": rfe.transform(X_cv_train), "y_train": y_cv_train,
        "X_val": rfe.transform(X_cv_val), "y_val": y_cv_val,
        "X_test": rfe.transform(X_test), "val_idx": cv_val_idx
    })

# Optunaの目的関数（feature_fraction → max_depth に変更）
def objective(trial):
    params = {
        "random_state": 42,
        "verbose": -1,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12), # 追加：木の最大深さ
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    scores = []
    for data in cv_datasets:
        model = LGBMRegressor(**params)
        model.fit(data["X_train"], data["y_train"])
        preds = np.clip(model.predict(data["X_val"]), 0, 5.00001)
        scores.append(np.sqrt(mean_squared_error(data["y_val"], preds)))
    return np.mean(scores)

# パラメータ探索の実行
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
best_params = study.best_params
print(f"\n★ ベストパラメータ: {best_params}")

# 4. 最適パラメータを用いて最終的な予測（OOFとテスト）を作成
oof_predictions = np.zeros(len(X_all_train))
test_predictions = np.zeros(len(X_test))
final_params = {"random_state": 42, "verbose": -1, **best_params}

for data in cv_datasets:
    model = LGBMRegressor(**final_params)
    model.fit(data["X_train"], data["y_train"])

    oof_predictions[data["val_idx"]] = np.clip(model.predict(data["X_val"]), 0, 5.00001)
    test_predictions += np.clip(model.predict(data["X_test"]), 0, 5.00001) / folds_num

# 5. スコア評価と提出ファイル作成
overall_oof_score = np.sqrt(mean_squared_error(y_all_train, oof_predictions))
print(f"全体のOOFスコア (RMSE): {overall_oof_score:.4f}")

sample["Price"] = test_predictions
# sample.to_csv('/content/drive/MyDrive/house_price_competion_hoshino/submission_optuna.csv', index=False)


RFEによる特徴量選択を開始します...


[I 2026-05-21 06:54:32,393] A new study created in memory with name: no-name-e5390634-9729-4618-8d0c-4138dfa643a9
[I 2026-05-21 06:54:34,175] Trial 0 finished with value: 0.4527493857281536 and parameters: {'lambda_l1': 2.3891890755179198e-06, 'lambda_l2': 0.003268673201657822, 'num_leaves': 56, 'max_depth': 9, 'bagging_fraction': 0.7270186176707762, 'bagging_freq': 2, 'min_child_samples': 78}. Best is trial 0 with value: 0.4527493857281536.
[I 2026-05-21 06:54:36,168] Trial 1 finished with value: 0.4422806905994081 and parameters: {'lambda_l1': 2.1178680075959206e-08, 'lambda_l2': 0.00037345236574198214, 'num_leaves': 70, 'max_depth': 12, 'bagging_fraction': 0.9584310634171881, 'bagging_freq': 1, 'min_child_samples': 49}. Best is trial 1 with value: 0.4422806905994081.
[I 2026-05-21 06:54:40,170] Trial 2 finished with value: 0.4462466473438097 and parameters: {'lambda_l1': 0.001473455384278604, 'lambda_l2': 1.0160965925245927e-06, 'num_leaves': 211, 'max_depth': 11, 'bagging_fraction'


★ ベストパラメータ: {'lambda_l1': 4.890094087336637e-05, 'lambda_l2': 3.458995503983615e-07, 'num_leaves': 89, 'max_depth': 10, 'bagging_fraction': 0.9341899691615144, 'bagging_freq': 5, 'min_child_samples': 19}
全体のOOFスコア (RMSE): 0.4416
